# Magic Hour · GHOST 2.0 Head Swap

Internal demo for **one-shot head swap** (not Krea2 image editing).

| Input | Role |
| --- | --- |
| **Body** | Scene — pose, clothing, background, expression driver |
| **Face** | Identity — head / face to transfer |

Upstream: [ai-forever/ghost-2.0](https://github.com/ai-forever/ghost-2.0)

**Recommended workflow:** edit §1 → **Runtime → Run all**.

**Expected runtime (warm, after models loaded)**

| GPU | Typical wall time |
| --- | --- |
| **A100** (40 GB) | ~30–90 s / image |
| **T4** (16 GB) | ~2–4 min / image |

Cold start downloads ~4 GB+ of checkpoints to Drive (cached for reconnects).

Warm re-run: change knobs or re-upload → re-run **§5** (and §6 for preview).


---
## 1 · Settings

Only these knobs are meant to change. Everything else uses GHOST 2.0 defaults.

To **reproduce** a prior run: open that run’s `run_config.json` and copy the knobs (and optional `PINNED_COMMIT`) below.


In [ ]:
# === User-facing knobs ===
OUTPUT_LONG_SIDE = 1024       # body long side before swap (px); 0 = native
USE_KANDI = False             # optional Kandinsky post-inpaint (slow / VRAM heavy)
DEBUG = False                 # verbose logs + keep work files

# Multi-person body photos: which face/head to swap
# largest | rightmost | leftmost | index (use BODY_FACE_INDEX)
BODY_FACE_POLICY = "largest"
BODY_FACE_INDEX = 0

ENABLE_MULTI_FACE_UI = False  # keep False for current product

# Quality gates (post-run warning thresholds)
IDENTITY_THRESH = 0.35
BODY_PSNR_THRESH = 28.0

# Optional: pin exact headswap_V2 commit (None = whatever git pull fetched)
PINNED_COMMIT = None

print("Settings")
print(f"  OUTPUT_LONG_SIDE={OUTPUT_LONG_SIDE}  USE_KANDI={USE_KANDI}  DEBUG={DEBUG}")
print(f"  BODY_FACE_POLICY={BODY_FACE_POLICY}  BODY_FACE_INDEX={BODY_FACE_INDEX}")
print(f"  ENABLE_MULTI_FACE_UI={ENABLE_MULTI_FACE_UI}")
print(f"  IDENTITY_THRESH={IDENTITY_THRESH}  BODY_PSNR_THRESH={BODY_PSNR_THRESH}")
print(f"  PINNED_COMMIT={PINNED_COMMIT or '(none — use pulled HEAD)'}")


---
## 2 · Setup _(first run / reconnect — collapsed by default)_

Mounts Drive, syncs this repo, clones GHOST 2.0 + deps, downloads/verifies checkpoints to Drive.

> First run can take 15–30+ minutes (large checkpoints). Later reconnects reuse Drive cache.


In [ ]:
#@title Setup: GPU · Drive · GHOST 2.0
from pathlib import Path
import importlib.util
import os
import subprocess

assert Path("/content").exists(), "This notebook is for Google Colab (/content)."

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime → Change runtime type → GPU (prefer A100), then Run all."
    )
print(f"✓ GPU  {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB)")

from google.colab import drive
print("→ Mounting Google Drive…")
drive.mount("/content/drive")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
print("→ Syncing headswap_V2…")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)

pin = globals().get("PINNED_COMMIT")
if pin:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", str(pin)], check=False)
    subprocess.run(["git", "checkout", str(pin)], check=False)

!pip install -q -e .

spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

os.environ["GHOST2_ROOT"] = "/content/ghost-2.0"
os.environ["GHOST2_DRIVE_ROOT"] = "/content/drive/MyDrive/headswap_ghost2"
os.environ["HEADSWAP_REPO"] = str(REPO)

print("→ Running GHOST 2.0 bootstrap (clone + weights)…")
!bash scripts/setup_ghost2_colab.sh

print("✓ Setup done. If Aligner import fails on first pass, Runtime → Restart session, then Run all.")


---
## 3 · Upload body & face

JPG / PNG / WEBP. Faces are validated.

If several faces are found in the body, the face is selected by **BODY_FACE_POLICY** from §1.


In [ ]:
import importlib.util
from pathlib import Path

from google.colab import files
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

custom = REPO / "data" / "custom"
custom.mkdir(parents=True, exist_ok=True)
cache_dir = REPO / ".cache" / "headswap_v2"
cache_dir.mkdir(parents=True, exist_ok=True)

BODY_FACE_POLICY = str(globals().get("BODY_FACE_POLICY", "largest"))
BODY_FACE_INDEX = int(globals().get("BODY_FACE_INDEX", 0))

try:
    print("Upload BODY image (scene / clothing / pose)…")
    up_body = files.upload()
    if not up_body:
        raise colab_demo.DemoError("No body image uploaded.")
    body_name = next(iter(up_body))
    body_path = custom / "body.png"
    body_im = colab_demo.save_upload(up_body[body_name], body_path)
    for msg in colab_demo.check_image_geometry(body_im, "Body"):
        colab_demo.warn(msg)
    body_face = colab_demo.require_face(body_im, cache_dir, "body")

    print("Upload FACE image (identity / head donor)…")
    up_face = files.upload()
    if not up_face:
        raise colab_demo.DemoError("No face image uploaded.")
    face_name = next(iter(up_face))
    face_path = custom / "face.png"
    face_im = colab_demo.save_upload(up_face[face_name], face_path)
    for msg in colab_demo.check_image_geometry(face_im, "Face"):
        colab_demo.warn(msg)
    face_face = colab_demo.require_face(face_im, cache_dir, "face")
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise SystemExit(str(exc))

display(Markdown("### Inputs"))
print(f"Body: {body_face['face_count']} face(s), size={body_im.size}, policy={BODY_FACE_POLICY}")
display(body_im.resize((320, 320)))
print(f"Face: {face_face['face_count']} face(s), conf={face_face['confidence']}  size={face_im.size}")
display(face_im.resize((320, 320)))
colab_demo.ok(f"Saved → {body_path} , {face_path}")


---
## 4 · Preflight _(collapsed)_

Confirms GPU, GHOST checkout, checkpoints, and detectable faces.


In [ ]:
#@title Preflight + environment summary
import importlib.util
import os
from pathlib import Path

REPO = Path("/content/headswap_V2")
GHOST = Path(os.environ.get("GHOST2_ROOT", "/content/ghost-2.0"))

spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

OUTPUT_LONG_SIDE = int(globals().get("OUTPUT_LONG_SIDE", 1024))
USE_KANDI = bool(globals().get("USE_KANDI", False))
DEBUG = bool(globals().get("DEBUG", False))
IDENTITY_THRESH = float(globals().get("IDENTITY_THRESH", 0.35))
BODY_PSNR_THRESH = float(globals().get("BODY_PSNR_THRESH", 28.0))
BODY_FACE_POLICY = str(globals().get("BODY_FACE_POLICY", "largest"))
BODY_FACE_INDEX = int(globals().get("BODY_FACE_INDEX", 0))
PINNED_COMMIT = globals().get("PINNED_COMMIT", None)

required = [
    GHOST / "aligner_checkpoints" / "aligner_1020_gaze_final.ckpt",
    GHOST / "blender_checkpoints" / "blender_lama.ckpt",
    GHOST / "weights" / "segformer_B5_ce.onnx",
    GHOST / "repos" / "stylematte" / "stylematte" / "checkpoints" / "stylematte_synth.pth",
]
try:
    gpu = colab_demo.verify_gpu()
    VERSIONS = colab_demo.collect_versions(repo=REPO, comfyui=None, pinned_commit=PINNED_COMMIT)
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise colab_demo.DemoError("Missing GHOST checkpoints:\n  - " + "\n  - ".join(missing))
    body_path = REPO / "data" / "custom" / "body.png"
    face_path = REPO / "data" / "custom" / "face.png"
    from PIL import Image
    body_im = Image.open(body_path).convert("RGB")
    face_im = Image.open(face_path).convert("RGB")
    cache_dir = REPO / ".cache" / "headswap_v2"
    colab_demo.require_face(body_im, cache_dir, "body")
    colab_demo.require_face(face_im, cache_dir, "face")
    PARAMS = {
        "pipeline": "ghost2_head_swap",
        "output_long_side": OUTPUT_LONG_SIDE,
        "use_kandi": USE_KANDI,
        "body_face_policy": BODY_FACE_POLICY,
        "body_face_index": BODY_FACE_INDEX,
        "debug": DEBUG,
        "identity_thresh": IDENTITY_THRESH,
        "body_psnr_thresh": BODY_PSNR_THRESH,
        "ghost2_root": str(GHOST),
        "repo_commit": (VERSIONS.get("git") or {}).get("commit"),
    }
    print("✓ Preflight OK")
    print(f"  GPU={gpu.get('name')}  ghost={GHOST}")
    for k, v in PARAMS.items():
        print(f"  {k}={v}")
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise SystemExit(str(exc))


---
## 5 · Run inference

Models load once per process. Each run writes a timestamped package under `/content/headswap_outputs/run_YYYYMMDD_HHMMSS/`.


In [ ]:
import importlib.util
import json
import os
import time
import traceback
from pathlib import Path

from PIL import Image

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

from headswap.config import load_config
from headswap.pipelines import create_pipeline

OUTPUT_LONG_SIDE = int(globals().get("OUTPUT_LONG_SIDE", 1024))
USE_KANDI = bool(globals().get("USE_KANDI", False))
DEBUG = bool(globals().get("DEBUG", False))
BODY_FACE_POLICY = str(globals().get("BODY_FACE_POLICY", "largest"))
BODY_FACE_INDEX = int(globals().get("BODY_FACE_INDEX", 0))
IDENTITY_THRESH = float(globals().get("IDENTITY_THRESH", 0.35))
BODY_PSNR_THRESH = float(globals().get("BODY_PSNR_THRESH", 28.0))

os.environ.setdefault("GHOST2_ROOT", "/content/ghost-2.0")
cfg = load_config(REPO / "configs" / "ghost2.yaml")
cfg.update({
    "ghost2_root": os.environ["GHOST2_ROOT"],
    "output_long_side": OUTPUT_LONG_SIDE,
    "use_kandi": USE_KANDI,
    "body_face_policy": BODY_FACE_POLICY,
    "body_face_index": BODY_FACE_INDEX,
    "save_debug": DEBUG,
    "verbose": DEBUG,
})

body = Image.open(REPO / "data" / "custom" / "body.png").convert("RGB")
face = Image.open(REPO / "data" / "custom" / "face.png").convert("RGB")
run_dir = colab_demo.make_run_dir("/content/headswap_outputs")
t0 = time.perf_counter()

try:
    pipe = create_pipeline(cfg)
    result = pipe.run(body, face, out_dir=run_dir)
    out = result.image
    latency = float(result.latency_s)
    meta = dict(result.meta or {})
    quality = colab_demo.score_result(
        body=body,
        face=face,
        result=out,
        latency_s=latency,
        cache_dir=REPO / ".cache" / "headswap_v2",
        pipeline="ghost2_head_swap",
        identity_thresh=IDENTITY_THRESH,
        body_psnr_thresh=BODY_PSNR_THRESH,
        stitch=True,
    )
    package = colab_demo.save_output_package(
        run_dir,
        result_image=out,
        run_config={
            "pipeline": "ghost2_head_swap",
            "knobs": {
                "OUTPUT_LONG_SIDE": OUTPUT_LONG_SIDE,
                "USE_KANDI": USE_KANDI,
                "DEBUG": DEBUG,
                "BODY_FACE_POLICY": BODY_FACE_POLICY,
                "BODY_FACE_INDEX": BODY_FACE_INDEX,
                "IDENTITY_THRESH": IDENTITY_THRESH,
                "BODY_PSNR_THRESH": BODY_PSNR_THRESH,
            },
            "meta": meta,
            "latency_s": latency,
        },
        metrics=quality,
        timing={"total_s": latency, "sampling_s": latency},
        debug_paths=result.debug_paths,
        save_debug=DEBUG,
    )
    RESULT_IMAGE = out
    RUN_DIR = run_dir
    QUALITY = quality
    LATENCY_S = latency
    PACKAGE = package
    print(f"✓ Done in {latency:.1f}s → {run_dir}")
    colab_demo.print_quality_report(quality)
except Exception as exc:
    colab_demo.fail(f"GHOST 2.0 run failed: {exc}")
    if DEBUG:
        traceback.print_exc()
    raise SystemExit(str(exc))



---
## 6 · Results


In [ ]:
from IPython.display import display, Markdown
from PIL import Image
from pathlib import Path
import importlib.util

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

body = Image.open(REPO / "data" / "custom" / "body.png").convert("RGB")
face = Image.open(REPO / "data" / "custom" / "face.png").convert("RGB")
result = globals().get("RESULT_IMAGE")
if result is None:
    raise SystemExit("No RESULT_IMAGE — run §5 first.")

display(Markdown("### Side-by-side"))
display(colab_demo.show_side_by_side(body, face, result))
display(Markdown("### Full result"))
display(result)
print(f"Run dir: {globals().get('RUN_DIR')}")
print(f"Stable: /content/headswap_outputs/HEADSWAP_RESULT.png")



---
## 7 · Run summary


In [ ]:
import importlib.util
from pathlib import Path

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

result = globals().get("RESULT_IMAGE")
run_dir = globals().get("RUN_DIR")
latency = float(globals().get("LATENCY_S") or 0)
quality = globals().get("QUALITY") or {}

colab_demo.print_run_summary(
    success=result is not None,
    total_s=latency,
    sampling_s=latency,
    steps=None,
    seed=None,
    gpu=None,
    resolution=list(result.size) if result is not None else None,
    output_path=(Path(run_dir) / "result.png") if run_dir else None,
    quality=quality,
)
print(f"  policy            {globals().get('BODY_FACE_POLICY')}")
print(f"  use_kandi         {globals().get('USE_KANDI')}")



---
## Notes

| Item | Detail |
| --- | --- |
| Model | GHOST 2.0 Aligner + Blender (head swap) |
| Source | Face / identity image |
| Target | Body / scene image |
| Multi-person | `BODY_FACE_POLICY` selects which detected face to drive |
| Hairstyle | Head swap **replaces** hair with the donor’s (by design) |
| Outputs | `result.png`, `run_config.json`, metrics under `/content/headswap_outputs/` |
| Stable shortcut | `/content/headswap_outputs/HEADSWAP_RESULT.png` |
| Upstream | https://github.com/ai-forever/ghost-2.0 |

Ethics: only use with consent; not for non-consensual deepfakes.
